# WildfireSpreadTS — Temporal Transformer Experiments

Implements **Architecture A** from the design doc, running **3-fold cross-validation**.

**Fixes vs `ViT_Unet.ipynb`:**
1. `ModelCheckpoint(monitor='val_loss')` instead of `val_f1`
2. 3-fold CV (`data_fold_id` ∈ {0, 3, 10}) instead of 1 fold
3. Proper temporal model (MultiScaleTemporalTransformer) instead of channel flatten

**Sections:**
0. Colab Setup · 1. Config · 2. Data Exploration · 3. Architecture · 4. Training · 5. Experiments · 6. Results

## 0. Colab Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


In [ ]:
import os

# Clone the WildfireSpreadTS repo for dataloader & UTAE baseline
if not os.path.exists('/content/WildfireSpreadTS'):
    !git clone --quiet https://github.com/SebastianGer/WildfireSpreadTS.git /content/WildfireSpreadTS

# Install dependencies — same versions as reference notebook
!pip install -q segmentation-models-pytorch pytorch-lightning==2.0.1 \
    torchmetrics einops jsonargparse[signatures]==4.20.1 wandb h5py rasterio \
    vector-quantize-pytorch

import sys
sys.path.insert(0, '/content/WildfireSpreadTS')
sys.path.insert(0, '/content/WildfireSpreadTS/src')

# Disable wandb (log locally only)
os.environ['WANDB_MODE'] = 'disabled'
os.environ['HDF5_USE_FILE_LOCKING'] = 'FALSE'

# Patch T_co import removed in newer torch versions
!sed -i 's/from torch.utils.data.dataset import T_co/T_co = None/' \
    /content/WildfireSpreadTS/src/dataloader/FireSpreadDataset.py

print("Setup complete.")


In [ ]:
import os, json, shutil, warnings
from glob import glob
from collections import defaultdict
from pathlib import Path

import h5py
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.gridspec import GridSpec
from matplotlib.patches import Rectangle
from tqdm.notebook import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, Callback
from IPython.display import display, clear_output
from torchmetrics import AveragePrecision

warnings.filterwarnings("ignore")
torch.set_float32_matmul_precision('medium')   # use Tensor Cores on A100/A10G

plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.size'] = 10

# Repo imports
from dataloader.FireSpreadDataModule import FireSpreadDataModule
from dataloader.FireSpreadDataset import FireSpreadDataset
from dataloader.utils import get_means_stds_missing_values

print(f"PyTorch {torch.__version__}, Lightning {pl.__version__}")
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None (CPU)'}")


In [ ]:
# ── Paths ────────────────────────────────────────────────────────────────────
# Point this at the folder on your Drive that contains the WildfireSpreadTS HDF5 files.
# Supports two layouts:
#   A) flat  : DATA_DIR/{year}/*.hdf5          (standard repo layout)
#   B) split : DATA_DIR/{split}/{year}/*.hdf5  (if you stored train/val/test separately)
DATA_DIR = "/content/drive/MyDrive/Climate Change - Project/WildfireSpreadTS"   # <── edit if needed

FLAT_DATA_DIR  = "/content/data_flat"    # symlink layer  (created below if needed)
LOCAL_DATA_DIR = "/content/data_local"  # local SSD copy (fast I/O during training)
RESULTS_DIR    = "/content/drive/MyDrive/Climate Change - Project/results"
os.makedirs(RESULTS_DIR, exist_ok=True)


In [ ]:
# ── Create flat year-level directory (needed by FireSpreadDataModule) ─────────
# If data is in split/year layout, create symlinks into FLAT_DATA_DIR/{year}/
# If already flat (year dirs at top level), use as-is.

os.makedirs(FLAT_DATA_DIR, exist_ok=True)

year_dirs_at_top = [d for d in os.listdir(DATA_DIR)
                    if d.isdigit() and os.path.isdir(os.path.join(DATA_DIR, d))]

if year_dirs_at_top:
    # Already flat — just point FLAT_DATA_DIR to DATA_DIR
    print("Data already in flat year-level layout.")
    FLAT_DATA_DIR = DATA_DIR
else:
    # split/year layout — create symlinks (same approach as reference notebook)
    for split in ['train', 'val', 'test']:
        split_dir = os.path.join(DATA_DIR, split)
        if not os.path.exists(split_dir):
            continue
        for year in os.listdir(split_dir):
            src = os.path.join(split_dir, year)
            dst = os.path.join(FLAT_DATA_DIR, year)
            if not os.path.exists(dst):
                os.symlink(src, dst)
                print(f"  symlink: {split}/{year} -> {dst}")

for year in sorted(os.listdir(FLAT_DATA_DIR)):
    n = len(glob(os.path.join(FLAT_DATA_DIR, year, '*.hdf5')))
    print(f"  {year}: {n} fires")


In [ ]:
# ── Copy HDF5 files to local SSD (avoids Drive FUSE timeouts during training) ─
# Matches the approach from the reference notebook.

all_files = []
for year in sorted(os.listdir(FLAT_DATA_DIR)):
    src_year = os.path.join(FLAT_DATA_DIR, year)
    dst_year = os.path.join(LOCAL_DATA_DIR, year)
    for f in sorted(glob(os.path.join(src_year, '*.hdf5'))):
        all_files.append((f, os.path.join(dst_year, os.path.basename(f)), dst_year))

already_copied = all(os.path.exists(dst) for _, dst, _ in all_files) and len(all_files) > 0

if already_copied:
    print("Files already copied to local disk.")
else:
    print(f"Copying {len(all_files)} files to local disk...")
    for src, dst, dst_dir in tqdm(all_files, desc="Copying to local disk"):
        os.makedirs(dst_dir, exist_ok=True)
        if not os.path.exists(dst):
            shutil.copy2(src, dst)

for year in sorted(os.listdir(LOCAL_DATA_DIR)):
    n = len([f for f in os.listdir(os.path.join(LOCAL_DATA_DIR, year)) if f.endswith('.hdf5')])
    print(f"  {year}: {n} fires")

DATA_DIR = LOCAL_DATA_DIR
print(f"\nDATA_DIR = {DATA_DIR}")


## 1. Configuration

In [ ]:
# ── 3-Fold Cross-Validation ──────────────────────────────────────────────────
# User spec:
#   Fold A: train=[2018,2019], val=2020, test=2021  → data_fold_id=0
#   Fold B: train=[2018,2020], val=2021, test=2019  → data_fold_id=3
#   Fold C: train=[2020,2021], val=2018, test=2019  → data_fold_id=10
FOLD_IDS   = [0, 3, 10]
FOLD_NAMES = [
    "A  train[18,19] / val[20] / test[21]",
    "B  train[18,20] / val[21] / test[19]",
    "C  train[20,21] / val[18] / test[19]",
]

# ── Raw feature names (23 channels, before one-hot landcover expansion) ───────
# Matches reference notebook FEATURE_NAMES exactly
FEATURE_NAMES = [
    'VIIRS band M11',        # 0
    'VIIRS band I2',         # 1
    'VIIRS band I1',         # 2
    'NDVI',                  # 3
    'EVI2',                  # 4
    'Total precipitation',   # 5
    'Wind speed',            # 6
    'Wind direction',        # 7
    'Min temperature',       # 8
    'Max temperature',       # 9
    'Energy release comp.',  # 10
    'Specific humidity',     # 11
    'Slope',                 # 12
    'Aspect',                # 13
    'Elevation',             # 14
    'Drought index (PDSI)',  # 15
    'Landcover class',       # 16
    'Fcast: Precipitation',  # 17
    'Fcast: Wind speed',     # 18
    'Fcast: Wind direction', # 19
    'Fcast: Temperature',    # 20
    'Fcast: Humidity',       # 21
    'Active fire',           # 22
]
FEATURE_GROUPS = {
    'Vegetation':   [0, 1, 2, 3, 4],
    'Weather':      [5, 6, 7, 8, 9, 11],
    'Topography':   [12, 13, 14],
    'Fire indices': [10, 15],
    'Landcover':    [16],
    'Forecast':     [17, 18, 19, 20, 21],
    'Active fire':  [22],
}

# ── Channel split for Architecture A (40-ch post-processing tensor) ───────────
STATIC_IDXS  = [3, 4, 12, 13, 14] + list(range(16, 33))   # 22 ch: NDVI,EVI2,topo,landcover
DYNAMIC_IDXS = [i for i in range(40) if i not in STATIC_IDXS]  # 18 ch

N_STATIC  = len(STATIC_IDXS)   # 22
N_DYNAMIC = len(DYNAMIC_IDXS)  # 18

# ── Hyperparameters ───────────────────────────────────────────────────────────
N_DAYS       = 5      # T: input sequence length  (ablate: 1, 3, 5)
BATCH_SIZE   = 32     # paper uses 32; reduce to 16/8 if OOM on T4
CROP_SIZE    = 128
MAX_EPOCHS   = 20     # fixed epoch budget — same for all models (fair comparison on 10% dataset)
LR           = 1e-3
D_STATIC     = 64
ENC_CHANNELS = [64, 128, 256, 512]
DEC_CHANNELS = [256, 128, 64, 32]
N_HEADS      = 8
FNO_MODES    = [32, 16, 8, 4]
NUM_WORKERS  = 8

print(f"Static channels : {N_STATIC}  {[FEATURE_NAMES[i] if i < 23 else f'LC_{i-16}' for i in STATIC_IDXS[:5]]}...")
print(f"Dynamic channels: {N_DYNAMIC}")
print(f"Folds: {list(zip(FOLD_IDS, [f.split()[0] for f in FOLD_NAMES]))}")


## 2. Data Exploration

### 2.1 Load & inventory the dataset

In [ ]:
def load_fire(path):
    """Load a single HDF5 fire file via local tmp copy to avoid Drive FUSE timeouts."""
    local_tmp = f"/content/tmp_{os.path.basename(path)}"
    shutil.copy2(path, local_tmp)
    with h5py.File(local_tmp, 'r') as hf:
        data = hf['data'][:]
    os.remove(local_tmp)
    return data   # shape: (T, 23, H, W)


# Build inventory by scanning year directories
inventory = []
for year in sorted(os.listdir(DATA_DIR)):
    year_dir = os.path.join(DATA_DIR, year)
    if not os.path.isdir(year_dir):
        continue
    files = sorted(glob(os.path.join(year_dir, '*.hdf5')))
    for f in files:
        try:
            with h5py.File(f, 'r') as hf:
                shape = hf['data'].shape
            inventory.append({
                'year': year, 'name': os.path.basename(f), 'path': f,
                'timesteps': shape[0], 'features': shape[1],
                'H': shape[2], 'W': shape[3],
            })
        except Exception as e:
            print(f"  Skipped {os.path.basename(f)}: {e}")

print(f"Total fires: {len(inventory)}")
for year in sorted(set(x['year'] for x in inventory)):
    items = [x for x in inventory if x['year'] == year]
    ts = sum(x['timesteps'] for x in items)
    print(f"  {year}: {len(items):3d} fires, {ts:5d} timesteps")


### 2.2 Dataset overview: fire durations & spatial sizes

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
year_colors = {'2018': '#2196F3', '2019': '#F44336', '2020': '#4CAF50', '2021': '#FF9800'}

# Fire duration histogram
for year, col in year_colors.items():
    d = [x['timesteps'] for x in inventory if x['year'] == year]
    axes[0].hist(d, bins=20, alpha=0.6, label=year, color=col)
axes[0].set_xlabel('Timesteps (days)'); axes[0].set_ylabel('Number of fires')
axes[0].set_title('Fire duration distribution'); axes[0].legend()

# Spatial dimensions
heights = [x['H'] for x in inventory]
widths  = [x['W'] for x in inventory]
years   = [x['year'] for x in inventory]
cols_pts = [year_colors.get(y, 'gray') for y in years]
axes[1].scatter(widths, heights, alpha=0.5, s=20, c=cols_pts)
axes[1].set_xlabel('Width (pixels)'); axes[1].set_ylabel('Height (pixels)')
axes[1].set_title('Spatial dimensions')

# Fires per year
year_counts = defaultdict(int)
for x in inventory:
    year_counts[x['year']] += 1
years_sorted = sorted(year_counts.keys())
bars = axes[2].bar(years_sorted, [year_counts[y] for y in years_sorted],
                   color=[year_colors.get(y, 'gray') for y in years_sorted])
axes[2].set_ylabel('Number of fires'); axes[2].set_title('Fires per year')
for bar, y in zip(bars, years_sorted):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 str(year_counts[y]), ha='center', fontsize=9)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'dataset_overview.png'), dpi=100, bbox_inches='tight')
plt.show()


### 2.3 Visualise a single fire: all 23 feature maps

In [ ]:
# Pick the longest fire for a rich visualisation
sample = sorted(inventory, key=lambda x: x['timesteps'], reverse=True)[0]
print(f"Showing: {sample['name']}  ({sample['year']})")
print(f"  Shape: {sample['timesteps']} timesteps x {sample['features']} features x {sample['H']}x{sample['W']}")

data = load_fire(sample['path'])
t = data.shape[0] // 2
print(f"  Displaying timestep {t}/{data.shape[0]-1}")


In [ ]:
# All 23 features at the selected timestep (matches reference notebook style)
n_features = len(FEATURE_NAMES)
cols = 6
rows = (n_features + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 2.8))
axes = axes.flatten()

for i in range(n_features):
    img = data[t, i]
    if np.nanmin(img) < 0:
        vabs = max(abs(np.nanmin(img)), abs(np.nanmax(img)))
        im = axes[i].imshow(img, cmap='RdBu_r', vmin=-vabs, vmax=vabs)
    elif i == 22:    # Active fire
        im = axes[i].imshow(img, cmap='hot')
    elif i == 16:    # Landcover (categorical)
        im = axes[i].imshow(img, cmap='tab20', interpolation='nearest')
    else:
        im = axes[i].imshow(img, cmap='viridis')
    axes[i].set_title(FEATURE_NAMES[i], fontsize=8)
    axes[i].axis('off')
    plt.colorbar(im, ax=axes[i], fraction=0.046, pad=0.04)

for i in range(n_features, len(axes)):
    axes[i].axis('off')

fig.suptitle(f"{sample['name']} — timestep {t}", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'feature_maps.png'), dpi=100, bbox_inches='tight')
plt.show()


### 2.4 Fire spread progression over time

In [ ]:
# Active fire mask + NDVI evolving across timesteps (matches reference)
n_show = min(12, data.shape[0])
step = max(1, data.shape[0] // n_show)
t_indices = list(range(0, data.shape[0], step))[:n_show]

fig, axes = plt.subplots(2, len(t_indices), figsize=(2.5 * len(t_indices), 5.5))

for col, t_idx in enumerate(t_indices):
    af_binary = (data[t_idx, 22] > 0).astype(float)
    axes[0, col].imshow(af_binary, cmap='Reds', vmin=0, vmax=1)
    axes[0, col].set_title(f'Day {t_idx}', fontsize=9)
    axes[0, col].axis('off')

    ndvi = data[t_idx, 3]
    axes[1, col].imshow(ndvi, cmap='YlGn',
                        vmin=np.nanpercentile(data[:, 3], 2),
                        vmax=np.nanpercentile(data[:, 3], 98))
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('Active fire', fontsize=10)
axes[1, 0].set_ylabel('NDVI', fontsize=10)
fig.suptitle(f'Fire progression: {sample["name"]}', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fire_progression.png'), dpi=100, bbox_inches='tight')
plt.show()


### 2.5 Cumulative fire spread map

In [ ]:
af_series = data[:, 22]   # (T, H, W)
first_detection = np.full(af_series.shape[1:], np.nan)
for t_idx in range(af_series.shape[0]):
    newly = (af_series[t_idx] > 0) & np.isnan(first_detection)
    first_detection[newly] = t_idx

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
cmap_fire = plt.cm.plasma.copy(); cmap_fire.set_bad('black')

im = axes[0].imshow(first_detection, cmap=cmap_fire)
plt.colorbar(im, ax=axes[0], label='Day of first detection')
axes[0].set_title('Cumulative fire spread'); axes[0].axis('off')

axes[1].imshow(data[0, 14], cmap='terrain')
fire_overlay = np.ma.masked_where(np.isnan(first_detection), first_detection)
axes[1].imshow(fire_overlay, cmap='Reds', alpha=0.6)
axes[1].set_title('Elevation + fire overlay'); axes[1].axis('off')

axes[2].imshow(data[0, 16], cmap='tab20', interpolation='nearest')
axes[2].imshow(fire_overlay, cmap='Reds', alpha=0.6)
axes[2].set_title('Landcover + fire overlay'); axes[2].axis('off')

fig.suptitle(f'{sample["name"]} — fire footprint vs terrain', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'cumulative_fire.png'), dpi=100, bbox_inches='tight')
plt.show()


### 2.6 Class imbalance: fire vs non-fire pixels

In [ ]:
fire_ratios = defaultdict(list)

for item in inventory:
    with h5py.File(item['path'], 'r') as hf:
        af = hf['data'][:, 22, :, :]
    fire_px  = np.nansum(af > 0)
    total_px = np.prod(af.shape) - int(np.isnan(af).sum())
    ratio = fire_px / total_px if total_px > 0 else 0
    fire_ratios[item['year']].append(ratio)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for year, col in year_colors.items():
    if fire_ratios[year]:
        axes[0].hist(fire_ratios[year], bins=25, alpha=0.6,
                     label=f'{year} (n={len(fire_ratios[year])})', color=col)
axes[0].set_xlabel('Fire pixel ratio per fire')
axes[0].set_ylabel('Count')
axes[0].set_title('Fire pixel ratio distribution by year')
axes[0].legend()

all_ratios = [r for v in fire_ratios.values() for r in v]
mean_ratio = np.mean(all_ratios)
axes[1].bar(['Non-fire', 'Fire'], [1 - mean_ratio, mean_ratio],
            color=['#78909C', '#FF5722'])
axes[1].set_ylabel('Fraction of pixels')
axes[1].set_title(f'Overall class balance  (fire = {mean_ratio:.4f})')
axes[1].set_yscale('log')

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'class_imbalance.png'), dpi=100, bbox_inches='tight')
plt.show()

print(f"Average fire pixel ratio : {mean_ratio:.5f}")
print(f"Implied pos_class_weight : {1/mean_ratio:.0f}")


### 2.7 Feature distributions

In [ ]:
# Sample pixels from several fires (matches reference sampling approach)
n_sample_fires = min(15, len(inventory))
rng = np.random.RandomState(42)
sample_indices = rng.choice(len(inventory), n_sample_fires, replace=False)

feature_samples = {i: [] for i in range(len(FEATURE_NAMES))}
for idx in sample_indices:
    with h5py.File(inventory[idx]['path'], 'r') as hf:
        d = hf['data'][:]
    for feat_i in range(d.shape[1]):
        vals = d[:, feat_i].flatten()
        vals = vals[~np.isnan(vals)]
        if len(vals) > 5000:
            vals = rng.choice(vals, 5000, replace=False)
        feature_samples[feat_i].append(vals)
for k in feature_samples:
    feature_samples[k] = np.concatenate(feature_samples[k])

cols = 6
rows = (len(FEATURE_NAMES) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(18, rows * 2.5))
axes = axes.flatten()

for i, name in enumerate(FEATURE_NAMES):
    vals = feature_samples[i]
    axes[i].hist(vals, bins=50, color='#5C6BC0', alpha=0.8, edgecolor='none')
    axes[i].set_title(name, fontsize=8)
    axes[i].tick_params(labelsize=7)
    axes[i].axvline(np.nanmean(vals), color='red', linewidth=0.8, linestyle='--')
for i in range(len(FEATURE_NAMES), len(axes)):
    axes[i].axis('off')

fig.suptitle('Feature distributions  (red = mean)', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'feature_distributions.png'), dpi=100, bbox_inches='tight')
plt.show()


### 2.8 Feature correlation heatmap

In [ ]:
n_feat   = len(FEATURE_NAMES)
min_len  = min(len(feature_samples[i]) for i in range(n_feat))
feat_mat = np.stack([feature_samples[i][:min_len] for i in range(n_feat)])
corr_mat = np.corrcoef(feat_mat)

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_mat, cmap='RdBu_r', vmin=-1, vmax=1)
ax.set_xticks(range(n_feat)); ax.set_yticks(range(n_feat))
ax.set_xticklabels(FEATURE_NAMES, rotation=45, ha='right', fontsize=7)
ax.set_yticklabels(FEATURE_NAMES, fontsize=7)
plt.colorbar(im, ax=ax, fraction=0.046)
ax.set_title('Feature correlation matrix')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'feature_correlation.png'), dpi=100, bbox_inches='tight')
plt.show()


### 2.9 Weather conditions: fire vs non-fire pixels

In [ ]:
fire_vals   = {i: [] for i in range(len(FEATURE_NAMES))}
nofire_vals = {i: [] for i in range(len(FEATURE_NAMES))}

for idx in sample_indices[:10]:
    with h5py.File(inventory[idx]['path'], 'r') as hf:
        d = hf['data'][:]
    fire_mask = d[:, 22] > 0
    for feat_i in range(d.shape[1]):
        fv  = d[:, feat_i][fire_mask];   fv  = fv[~np.isnan(fv)]
        nfv = d[:, feat_i][~fire_mask];  nfv = nfv[~np.isnan(nfv)]
        if len(fv)  > 2000: fv  = rng.choice(fv,  2000, replace=False)
        if len(nfv) > 2000: nfv = rng.choice(nfv, 2000, replace=False)
        fire_vals[feat_i].append(fv); nofire_vals[feat_i].append(nfv)

for k in fire_vals:
    fire_vals[k]   = np.concatenate(fire_vals[k])   if fire_vals[k]   else np.array([])
    nofire_vals[k] = np.concatenate(nofire_vals[k]) if nofire_vals[k] else np.array([])

compare_features = [5, 6, 8, 9, 10, 11, 3, 4]   # precip, wind, temps, ERC, humidity, NDVI, EVI2
fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for ax_i, feat_i in enumerate(compare_features):
    fv, nfv = fire_vals[feat_i], nofire_vals[feat_i]
    if not len(fv) or not len(nfv):
        axes[ax_i].set_title(f'{FEATURE_NAMES[feat_i]}\n(no data)'); continue
    lo = min(np.percentile(nfv, 1), np.percentile(fv, 1))
    hi = max(np.percentile(nfv, 99), np.percentile(fv, 99))
    bins = np.linspace(lo, hi, 40)
    axes[ax_i].hist(nfv, bins=bins, alpha=0.6, density=True, label='Non-fire', color='#78909C')
    axes[ax_i].hist(fv,  bins=bins, alpha=0.6, density=True, label='Fire',     color='#FF5722')
    axes[ax_i].set_title(FEATURE_NAMES[feat_i], fontsize=10)
    axes[ax_i].legend(fontsize=8); axes[ax_i].tick_params(labelsize=8)

fig.suptitle('Feature distributions: Fire pixels vs Non-fire pixels', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'fire_vs_nofire.png'), dpi=100, bbox_inches='tight')
plt.show()


### 2.10 Test the DataModule

In [ ]:
# Smoke-test the DataModule with fold A (data_fold_id=0)
# Note: return_doy=True because our WildfireTransformer and UTAE use day-of-year encoding
dm_test = FireSpreadDataModule(
    data_dir=DATA_DIR,
    batch_size=BATCH_SIZE,
    n_leading_observations=N_DAYS,
    n_leading_observations_test_adjustment=5,
    crop_side_length=CROP_SIZE,
    load_from_hdf5=True,
    num_workers=2,
    remove_duplicate_features=False,
    features_to_keep=None,
    return_doy=True,
    data_fold_id=0,
)
dm_test.setup('fit')
x_batch, y_batch, doys_batch = next(iter(dm_test.train_dataloader()))

print(f"Input  shape: {tuple(x_batch.shape)}  (B, T, C, H, W)")
print(f"Target shape: {tuple(y_batch.shape)}  (B, H, W)")
print(f"DOY    shape: {tuple(doys_batch.shape)}  (B, T)")
print(f"Target values: {y_batch.unique().tolist()}")
print(f"Fire pixel ratio in batch: {y_batch.float().mean():.4f}")
del dm_test


## 3. Architecture A — WildfireTransformer

### 3a. FiLM Conditioning + Static Branch

In [ ]:
class FiLM(nn.Module):
    """Feature-wise Linear Modulation: scale+shift a feature map using static features."""
    def __init__(self, static_dim, feature_dim):
        super().__init__()
        self.pool  = nn.AdaptiveAvgPool2d(1)
        self.gamma = nn.Linear(static_dim, feature_dim)
        self.beta  = nn.Linear(static_dim, feature_dim)

    def forward(self, x, static_feat):
        s = self.pool(static_feat).flatten(1)
        g = self.gamma(s).unsqueeze(-1).unsqueeze(-1)
        b = self.beta(s).unsqueeze(-1).unsqueeze(-1)
        return g * x + b


class StaticBranch(nn.Module):
    """
    Encodes 22 static channels → spatial embedding (B, out_dim, H, W).
    5 continuous: NDVI, EVI2, slope, aspect(sin), elevation
    17 categorical: landcover one-hot (already encoded by dataset)
    """
    def __init__(self, n_continuous=5, n_categorical=17, out_dim=D_STATIC):
        super().__init__()
        self.cont_enc = nn.Sequential(
            nn.Conv2d(n_continuous, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
            nn.Conv2d(32, 32, 3, padding=1),           nn.BatchNorm2d(32), nn.ReLU(True),
        )
        self.cat_enc = nn.Sequential(
            nn.Conv2d(n_categorical, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(True),
        )
        self.fuse = nn.Sequential(
            nn.Conv2d(64, out_dim, 1), nn.BatchNorm2d(out_dim), nn.ReLU(True),
        )

    def forward(self, static):
        return self.fuse(torch.cat([self.cont_enc(static[:, :5]),
                                    self.cat_enc(static[:, 5:])], dim=1))

_sb = StaticBranch()
print(f"StaticBranch: (2,22,128,128) -> {tuple(_sb(torch.randn(2,22,128,128)).shape)}  ✓")


### 3b. CNN Encoder (ResNet18-style + FiLM)

In [ ]:
class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, stride=1):
        super().__init__()
        self.c1 = nn.Conv2d(in_ch, out_ch, 3, stride=stride, padding=1, bias=False)
        self.b1 = nn.BatchNorm2d(out_ch)
        self.c2 = nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False)
        self.b2 = nn.BatchNorm2d(out_ch)
        self.skip = (nn.Sequential(nn.Conv2d(in_ch, out_ch, 1, stride=stride, bias=False),
                                   nn.BatchNorm2d(out_ch))
                     if stride != 1 or in_ch != out_ch else nn.Identity())

    def forward(self, x):
        return F.relu(self.b2(self.c2(F.relu(self.b1(self.c1(x))))) + self.skip(x), inplace=True)


class CNNEncoder(nn.Module):
    """ResNet18-based encoder shared across timesteps. FiLM at every scale."""
    def __init__(self, in_ch=N_DYNAMIC, film_dim=D_STATIC, widths=None):
        super().__init__()
        widths = widths or ENC_CHANNELS
        self.stem   = nn.Sequential(nn.Conv2d(in_ch, widths[0], 7, stride=2, padding=3, bias=False),
                                    nn.BatchNorm2d(widths[0]), nn.ReLU(True))
        self.layer1 = nn.Sequential(ResBlock(widths[0], widths[0]),    ResBlock(widths[0], widths[0]))
        self.layer2 = nn.Sequential(ResBlock(widths[0], widths[1], 2), ResBlock(widths[1], widths[1]))
        self.layer3 = nn.Sequential(ResBlock(widths[1], widths[2], 2), ResBlock(widths[2], widths[2]))
        self.layer4 = nn.Sequential(ResBlock(widths[2], widths[3], 2), ResBlock(widths[3], widths[3]))
        self.film1 = FiLM(film_dim, widths[0]); self.film2 = FiLM(film_dim, widths[1])
        self.film3 = FiLM(film_dim, widths[2]); self.film4 = FiLM(film_dim, widths[3])

    def forward(self, x_t, static_feat):
        x  = self.stem(x_t)
        f1 = self.film1(self.layer1(x),  F.adaptive_avg_pool2d(static_feat, x.shape[-2:]))
        f2 = self.film2(self.layer2(f1), F.adaptive_avg_pool2d(static_feat, (f1.shape[-2]//2, f1.shape[-1]//2)))
        f3 = self.film3(self.layer3(f2), F.adaptive_avg_pool2d(static_feat, (f2.shape[-2]//2, f2.shape[-1]//2)))
        f4 = self.film4(self.layer4(f3), F.adaptive_avg_pool2d(static_feat, (f3.shape[-2]//2, f3.shape[-1]//2)))
        return [f1, f2, f3, f4]

_enc = CNNEncoder(); _sf = _sb(torch.randn(2,22,128,128))
print("CNN Encoder scales:", [tuple(f.shape) for f in _enc(torch.randn(2,N_DYNAMIC,128,128), _sf)])


### 3c. FNO Encoder (Fourier Neural Operator — from scratch)

In [ ]:
class SpectralConv2d(nn.Module):
    def __init__(self, in_ch, out_ch, modes1, modes2):
        super().__init__()
        self.modes1, self.modes2 = modes1, modes2
        scale = 1.0 / (in_ch * out_ch)
        self.w1 = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes1, modes2, dtype=torch.cfloat))
        self.w2 = nn.Parameter(scale * torch.randn(in_ch, out_ch, modes1, modes2, dtype=torch.cfloat))

    def forward(self, x):
        B, C, H, W = x.shape
        xf = torch.fft.rfft2(x)
        out = torch.zeros(B, self.w1.shape[1], H, W//2+1, dtype=torch.cfloat, device=x.device)
        m1, m2 = min(self.modes1, H//2), min(self.modes2, W//4+1)
        out[:, :, :m1,  :m2] = torch.einsum("bixy,ioxy->boxy", xf[:, :, :m1,  :m2], self.w1[:,:,:m1,:m2])
        out[:, :, -m1:, :m2] = torch.einsum("bixy,ioxy->boxy", xf[:, :, -m1:, :m2], self.w2[:,:,:m1,:m2])
        return torch.fft.irfft2(out, s=(H, W))


class FNOBlock(nn.Module):
    def __init__(self, in_ch, out_ch, modes=16):
        super().__init__()
        self.spec  = SpectralConv2d(in_ch, out_ch, modes, modes)
        self.local = nn.Conv2d(in_ch, out_ch, 1)
        self.norm  = nn.InstanceNorm2d(out_ch, affine=True)
        self.act   = nn.GELU()
    def forward(self, x):
        return self.act(self.norm(self.spec(x) + self.local(x)))


class FNOEncoder(nn.Module):
    """Multi-scale FNO encoder. Static features concatenated to input."""
    def __init__(self, in_ch=None, widths=None, modes=None):
        super().__init__()
        in_ch  = in_ch  or (N_DYNAMIC + D_STATIC)
        widths = widths or ENC_CHANNELS
        modes  = modes  or FNO_MODES
        self.lift   = nn.Conv2d(in_ch, widths[0], 1)
        self.stages = nn.ModuleList()
        self.pools  = nn.ModuleList()
        for i, (w, m) in enumerate(zip(widths, modes)):
            in_w = widths[i-1] if i > 0 else widths[0]
            self.stages.append(nn.Sequential(FNOBlock(in_w, w, m), FNOBlock(w, w, m)))
            if i > 0:
                self.pools.append(nn.Conv2d(widths[i-1], widths[i-1], 3, stride=2, padding=1))

    def forward(self, x_t, static_feat):
        x = self.lift(torch.cat([x_t, static_feat], dim=1))
        feats = []
        for i, stage in enumerate(self.stages):
            if i > 0: x = self.pools[i-1](x)
            x = stage(x); feats.append(x)
        return feats

print(f"FNO params: {sum(p.numel() for p in FNOEncoder().parameters())/1e6:.2f}M")


### 3d. Multi-Scale Temporal Transformer

In [ ]:
class TemporalAttentionBlock(nn.Module):
    """
    Collapses T frames -> 1 via learnable-query attention.
    Applied at EVERY encoder scale (unlike UTAE which only uses bottleneck).
    """
    def __init__(self, dim, n_heads=N_HEADS):
        super().__init__()
        n_heads = max(1, min(n_heads, dim // 8))
        self.mha   = nn.MultiheadAttention(dim, n_heads, batch_first=True)
        self.norm  = nn.LayerNorm(dim)
        self.query = nn.Parameter(torch.randn(1, 1, dim) * 0.02)

    def forward(self, x):
        N, T, C = x.shape
        q = self.query.expand(N, 1, C)
        out, attn = self.mha(q, self.norm(x), self.norm(x))
        return out.squeeze(1), attn.squeeze(1)   # (N,C), (N,T)


class MultiScaleTemporalTransformer(nn.Module):
    def __init__(self, channels=None, n_heads=N_HEADS, max_T=10):
        super().__init__()
        channels = channels or ENC_CHANNELS
        self.temporal_attns = nn.ModuleList([TemporalAttentionBlock(ch, n_heads) for ch in channels])
        self.pos_embeds = nn.ParameterList(
            [nn.Parameter(torch.randn(1, max_T, ch) * 0.02) for ch in channels])

    def forward(self, multi_scale_features):
        fused, attn_maps = [], []
        for s, (feat_seq, attn_block) in enumerate(zip(multi_scale_features, self.temporal_attns)):
            B, T, C, H, W = feat_seq.shape
            feat_seq = feat_seq + self.pos_embeds[s][:, :T, :, None, None]
            x = feat_seq.permute(0, 3, 4, 1, 2).reshape(B*H*W, T, C)
            out, attn = attn_block(x)
            fused.append(out.reshape(B, H, W, C).permute(0, 3, 1, 2))
            attn_maps.append(attn.reshape(B, H, W, T))
        return fused, attn_maps

_T = N_DAYS
_ms = [torch.randn(2, _T, ENC_CHANNELS[i], 128//(2**(i+1)), 128//(2**(i+1))) for i in range(4)]
_fused, _ = MultiScaleTemporalTransformer()(_ms)
print("MSTT fused scales:", [tuple(f.shape) for f in _fused])


### 3e. U-Net Decoder + Probabilistic Head

In [ ]:
class UNetDecoder(nn.Module):
    def __init__(self, enc_ch=None, dec_ch=None):
        super().__init__()
        enc_ch = enc_ch or ENC_CHANNELS; dec_ch = dec_ch or DEC_CHANNELS
        self.up_blocks = nn.ModuleList()
        for i in range(len(dec_ch)):
            in_ch  = enc_ch[-(i+1)] if i == 0 else dec_ch[i-1] + enc_ch[-(i+1)]
            self.up_blocks.append(nn.Sequential(
                nn.ConvTranspose2d(in_ch, dec_ch[i], 2, stride=2),
                nn.BatchNorm2d(dec_ch[i]), nn.ReLU(True),
                nn.Conv2d(dec_ch[i], dec_ch[i], 3, padding=1),
                nn.BatchNorm2d(dec_ch[i]), nn.ReLU(True),
            ))
    def forward(self, feats):
        x = feats[0]
        for i, block in enumerate(self.up_blocks):
            if i > 0: x = torch.cat([x, feats[i]], dim=1)
            x = block(x)
        return x


class ProbabilisticHead(nn.Module):
    """Fire probability + aleatoric uncertainty (heteroscedastic output)."""
    def __init__(self, in_ch=None):
        super().__init__()
        in_ch = in_ch or DEC_CHANNELS[-1]
        self.mean_head = nn.Conv2d(in_ch, 1, 1)
        self.var_head  = nn.Conv2d(in_ch, 1, 1)
    def forward(self, x):
        return torch.sigmoid(self.mean_head(x)), self.var_head(x)

print("UNetDecoder + ProbabilisticHead defined.")


### 3f. WildfireTransformer — Full Architecture A

In [ ]:
class WildfireTransformer(nn.Module):
    """
    Architecture A: Static/Dynamic split + Encoder + MSTT + Decoder.
    encoder_type: "cnn"  (ResNet18 + FiLM)  |  "fno"  (Fourier Neural Operator)
    """
    def __init__(self, encoder_type="cnn", d_static=D_STATIC,
                 enc_channels=None, dec_channels=None, n_heads=N_HEADS):
        super().__init__()
        enc_channels = enc_channels or ENC_CHANNELS
        dec_channels = dec_channels or DEC_CHANNELS
        self.encoder_type  = encoder_type
        self.static_branch = StaticBranch(5, 17, d_static)
        if encoder_type == "cnn":
            self.encoder = CNNEncoder(N_DYNAMIC, d_static, enc_channels)
        elif encoder_type == "fno":
            self.encoder = FNOEncoder(N_DYNAMIC + d_static, enc_channels)
        else:
            raise ValueError(f"Unknown encoder_type: {encoder_type!r}")
        self.temporal_fusion = MultiScaleTemporalTransformer(enc_channels, n_heads)
        self.decoder         = UNetDecoder(enc_channels, dec_channels)
        self.head            = ProbabilisticHead(dec_channels[-1])

    def forward(self, x, doys=None):
        """x: (B,T,40,H,W)  doys: (B,T) ignored (handled by learned pos embeddings)"""
        B, T, _, H, W = x.shape
        static_in   = x[:, 0][:, STATIC_IDXS]     # (B,22,H,W)
        dynamic_seq = x[:, :, DYNAMIC_IDXS]        # (B,T,18,H,W)
        static_feat = self.static_branch(static_in)
        multi_scale = [[] for _ in range(4)]
        for t in range(T):
            feats = (self.encoder(dynamic_seq[:, t], static_feat)
                     if self.encoder_type == "cnn" else
                     self.encoder(dynamic_seq[:, t],
                                  F.adaptive_avg_pool2d(static_feat, dynamic_seq.shape[-2:])))
            for s in range(4): multi_scale[s].append(feats[s])
        multi_scale = [torch.stack(ms, dim=1) for ms in multi_scale]
        fused, attn_maps = self.temporal_fusion(multi_scale)
        pred_mean, pred_log_var = self.head(self.decoder([fused[3],fused[2],fused[1],fused[0]]))
        return pred_mean, pred_log_var, attn_maps


# Sanity checks
print("=== Architecture sanity checks ===")
for enc_type in ["cnn", "fno"]:
    mdl = WildfireTransformer(encoder_type=enc_type)
    _x  = torch.randn(2, N_DAYS, 40, 128, 128)
    with torch.no_grad():
        _pm, _plv, _am = mdl(_x)
    n_p = sum(p.numel() for p in mdl.parameters() if p.requires_grad)
    print(f"  {enc_type.upper()}: pred={tuple(_pm.shape)}  params={n_p/1e6:.2f}M")
    del mdl


## 4. Loss Functions

In [ ]:
EPS = 1e-6

def focal_bce_loss(pred, target, gamma=2.0, alpha=0.97):
    """Focal BCE for ~0.1% fire pixels. alpha≈inverse fire freq."""
    pred = pred.clamp(EPS, 1-EPS)
    bce  = -(target * torch.log(pred) + (1-target) * torch.log(1-pred))
    pt   = target * pred + (1-target) * (1-pred)
    return ((1-pt)**gamma * (target*alpha + (1-target)*(1-alpha)) * bce).mean()


def focal_heteroscedastic_loss(pred_mean, pred_log_var, target, gamma=2.0, alpha=0.97):
    """Focal + heteroscedastic: robust to noisy VIIRS labels."""
    pred_mean = pred_mean.clamp(EPS, 1-EPS)
    bce  = -(target * torch.log(pred_mean) + (1-target) * torch.log(1-pred_mean))
    het  = bce / (2 * torch.exp(pred_log_var) + EPS) + 0.5 * pred_log_var
    pt   = target * pred_mean + (1-target) * (1-pred_mean)
    return ((1-pt)**gamma * (target*alpha + (1-target)*(1-alpha)) * het).mean()


def soft_dice_loss(pred, target):
    """Differentiable Dice (matches paper baseline training)."""
    p, t = pred.reshape(-1), target.reshape(-1).float()
    return 1 - (2*(p*t).sum() + EPS) / (p.sum() + t.sum() + EPS)

# Quick check
_p = torch.rand(4,1,128,128); _lv = torch.randn(4,1,128,128); _t = torch.randint(0,2,(4,128,128)).float()
print(f"focal_bce:   {focal_bce_loss(_p.squeeze(1), _t):.4f}")
print(f"focal_het:   {focal_heteroscedastic_loss(_p, _lv, _t.unsqueeze(1)):.4f}")
print(f"soft_dice:   {soft_dice_loss(_p.squeeze(1), _t):.4f}")


## 5. Lightning Modules

### 5a. WildfireLightningModule (Architecture A)

In [ ]:
class LivePlotCallback(Callback):
    """Live loss/metric plot updated each validation epoch (matches reference style)."""
    def __init__(self):
        self.train_loss, self.val_loss, self.val_ap, self.epochs = [], [], [], []

    def on_validation_epoch_end(self, trainer, pl_module):
        m = trainer.callback_metrics
        self.epochs.append(trainer.current_epoch)
        self.train_loss.append(float(m.get('train_loss_epoch', float('nan'))))
        self.val_loss.append(float(m.get('val_loss', float('nan'))))
        self.val_ap.append(float(m.get('val_ap', float('nan'))))

        clear_output(wait=True)
        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        axes[0].plot(self.epochs, self.train_loss, 'o-', label='Train', color='#2196F3', markersize=4)
        axes[0].plot(self.epochs, self.val_loss,   'o-', label='Val',   color='#FF9800', markersize=4)
        axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)
        axes[1].plot(self.epochs, self.val_ap, 'o-', color='#4CAF50', markersize=4)
        axes[1].set_title('Val AP'); axes[1].grid(alpha=0.3)
        axes[2].axis('off')
        best_val = min(self.val_loss) if self.val_loss else 0
        best_ep  = self.epochs[self.val_loss.index(best_val)] if self.val_loss else 0
        axes[2].text(0.1, 0.5,
            f"Epoch: {trainer.current_epoch}\n\n"
            f"Train Loss: {self.train_loss[-1]:.4f}\n"
            f"Val   Loss: {self.val_loss[-1]:.4f}\n"
            f"Val   AP:   {self.val_ap[-1]:.4f}\n\n"
            f"{'─'*25}\n"
            f"Best Val Loss: {best_val:.4f}\n"
            f"  (epoch {best_ep})",
            transform=axes[2].transAxes, fontsize=11, va='center', family='monospace')
        plt.tight_layout(); plt.show()


class WildfireLightningModule(pl.LightningModule):
    """
    Lightning wrapper for WildfireTransformer.
    FIX vs reference: ModelCheckpoint monitors val_loss (not val_f1).
    """
    def __init__(self, model, loss_type="focal_het", lr=LR):
        super().__init__()
        self.model = model; self.loss_type = loss_type; self.lr = lr
        self.val_ap  = AveragePrecision(task="binary")
        self.test_ap = AveragePrecision(task="binary")

    def _loss(self, pm, plv, y):
        if self.loss_type == "focal_het": return focal_heteroscedastic_loss(pm, plv, y.unsqueeze(1))
        if self.loss_type == "focal":     return focal_bce_loss(pm.squeeze(1), y)
        if self.loss_type == "dice":      return soft_dice_loss(pm.squeeze(1), y)
        raise ValueError(self.loss_type)

    def forward(self, x, doys=None): return self.model(x, doys)

    def training_step(self, batch, _):
        x, y, doys = batch; pm, plv, _ = self(x, doys)
        loss = self._loss(pm, plv, y)
        self.log('train_loss', loss, on_step=True, on_epoch=True, prog_bar=True)
        return loss

    def validation_step(self, batch, _):
        x, y, doys = batch; pm, plv, _ = self(x, doys)
        loss = self._loss(pm, plv, y)
        self.log('val_loss', loss, on_epoch=True, prog_bar=True, sync_dist=True)
        self.val_ap.update(pm.squeeze(1).reshape(-1).float(), y.reshape(-1).int())
        return loss

    def on_validation_epoch_end(self):
        self.log('val_ap', self.val_ap.compute(), prog_bar=True); self.val_ap.reset()

    def test_step(self, batch, _):
        x, y, doys = batch; pm, plv, _ = self(x, doys)
        self.test_ap.update(pm.squeeze(1).reshape(-1).float(), y.reshape(-1).int())

    def on_test_epoch_end(self):
        ap = self.test_ap.compute(); self.log('test_ap', ap); self.test_ap.reset()
        print(f"  Test AP: {ap.item():.4f}")

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self.lr, betas=(0.9,0.999), weight_decay=0.01)
        sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=MAX_EPOCHS, eta_min=self.lr*0.01)
        return {"optimizer": opt, "lr_scheduler": {"scheduler": sched, "interval": "epoch"}}



### 5b. UTAE Baseline Wrapper (from existing repo)

In [ ]:
from models.UTAELightning import UTAELightning

class UTAEWrapper(UTAELightning):
    """
    Wraps existing UTAELightning with:
      - configure_optimizers (required for standalone training)
      - on_test_epoch_end    (removes wandb dependency)
    """
    def __init__(self, n_channels, pos_class_weight=10, lr=0.01, loss_fn="BCE"):
        # Paper config (cfgs/UTAE/all_features.yaml):
        #   loss_function = "BCE", pos_class_weight = 10, lr = 0.01, AdamW
        super().__init__(
            n_channels=n_channels,
            flatten_temporal_dimension=False,
            pos_class_weight=pos_class_weight,
            loss_function=loss_fn,
        )
        self._lr = lr

    def configure_optimizers(self):
        opt = torch.optim.AdamW(self.parameters(), lr=self._lr, betas=(0.9,0.999), weight_decay=0.01)
        return {"optimizer": opt}

    def on_test_epoch_end(self):
        ap = self.test_avg_precision.compute(); self.log('test_ap', ap)
        print(f"  Test AP (UTAE): {ap.item():.4f}")

print("UTAEWrapper defined.")


## 6. Experiment Runner

In [ ]:
def get_datamodule(fold_id, n_days=N_DAYS, batch_size=BATCH_SIZE):
    return FireSpreadDataModule(
        data_dir=DATA_DIR, batch_size=batch_size,
        n_leading_observations=n_days, n_leading_observations_test_adjustment=5,
        crop_side_length=CROP_SIZE, load_from_hdf5=True, num_workers=NUM_WORKERS,
        remove_duplicate_features=False, features_to_keep=None,
        return_doy=True, data_fold_id=fold_id,
    )


def run_experiment(name, model, fold_id, fold_name,
                   loss_type="focal_het", lr=LR, use_live_plot=True):
    """Train one model on one fold. Returns test AP (float).
    MAX_EPOCHS applies to all models equally — same training budget for fair comparison.
    """
    is_utae = isinstance(model, UTAEWrapper)
    ckpt_dir = os.path.join(RESULTS_DIR, name, f"fold_{fold_id}")
    os.makedirs(ckpt_dir, exist_ok=True)
    dm = get_datamodule(fold_id)

    lit = (model if is_utae
           else WildfireLightningModule(model, loss_type=loss_type, lr=lr))

    ckpt_cb = ModelCheckpoint(
        dirpath=ckpt_dir, filename='best',
        monitor='val_loss', mode='min', save_top_k=1,
    )
    callbacks = [ckpt_cb]
    if use_live_plot and not is_utae:
        callbacks.append(LivePlotCallback())

    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS,   # same for all models — fair comparison
        accelerator='auto',
        precision='32-true' if is_utae else ('16-mixed' if torch.cuda.is_available() else '32-true'),
        callbacks=callbacks, log_every_n_steps=10,
        enable_progress_bar=True, enable_model_summary=False, logger=False,
    )

    print(f"\n{'='*60}")
    print(f"Experiment : {name}")
    print(f"Fold       : {fold_id} — {fold_name}")
    print(f"{'='*60}")

    trainer.fit(lit, datamodule=dm)
    results = trainer.test(lit, datamodule=dm, ckpt_path='best')
    ap = float(results[0].get('test_ap', results[0].get('test_AP', 0.0)))

    out = {"name": name, "fold_id": fold_id, "fold_name": fold_name, "test_ap": ap,
           "ckpt": ckpt_cb.best_model_path}
    with open(os.path.join(ckpt_dir, 'result.json'), 'w') as f:
        json.dump(out, f, indent=2)
    return ap


def run_all(experiments):
    all_results = {}
    for exp in experiments:
        all_results[exp['name']] = {}
        for fold_id, fold_name in zip(FOLD_IDS, FOLD_NAMES):
            if exp['encoder'] == 'utae':
                # Exact paper hyperparameters (cfgs/UTAE/all_features.yaml):
                # loss="BCE", pos_class_weight=10, lr=0.01, AdamW, 32-bit
                model = UTAEWrapper(n_channels=40, pos_class_weight=10, lr=0.01, loss_fn="BCE")
            else:
                model = WildfireTransformer(encoder_type=exp['encoder'])
            ap = run_experiment(exp['name'], model, fold_id, fold_name,
                                loss_type=exp.get('loss', 'focal_het'))
            all_results[exp['name']][fold_id] = ap
    with open(os.path.join(RESULTS_DIR, 'all_results.json'), 'w') as f:
        json.dump(all_results, f, indent=2)
    return all_results

print("Experiment runner defined.")


## 7. Run Experiments

In [ ]:
# Runtime estimate per experiment × 3 folds on Colab A100/T4:
#   UTAE_baseline:    ~20 min  (small model, FP32)
#   E1/E3/L* (ArchA): ~40 min  (larger model, AMP)
# Total for all 7 experiments × 3 folds ≈ 5–10 h on T4.

EXPERIMENTS = [
    # ── Baselines ──────────────────────────────────────────────────────────────
    {"name": "UTAE_baseline",    "encoder": "utae", "loss": "BCE"},       # paper UTAE (all features)
    # ── Architecture A: encoder ablation (all use focal_het loss) ─────────────
    {"name": "E1_CNN_focal_het", "encoder": "cnn",  "loss": "focal_het"}, # Arch A + CNN
    {"name": "E3_FNO_focal_het", "encoder": "fno",  "loss": "focal_het"}, # Arch A + FNO
    # ── Architecture A: loss ablation (CNN encoder) ────────────────────────────
    {"name": "L1_CNN_dice",      "encoder": "cnn",  "loss": "dice"},      # Arch A + Dice
    {"name": "L2_CNN_focal",     "encoder": "cnn",  "loss": "focal"},     # Arch A + Focal-BCE
    # ── (Optional) feature ablation — uncomment to run ────────────────────────
    # {"name": "F1_CNN_veg_only",  "encoder": "cnn",  "loss": "focal_het", # vegetation + fire
    #  "features": [0,1,2,3,4,38,39]},
    # {"name": "F2_CNN_multi",     "encoder": "cnn",  "loss": "focal_het", # multi-modal
    #  "features": [0,1,2,3,4,5,6,7,8,9,11,12,13,14,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,38,39]},
]

# Quick smoke test (1 experiment, 1 fold, 100 steps):
# EXPERIMENTS = [{"name": "smoke_test", "encoder": "cnn", "loss": "focal_het"}]
# FOLD_IDS = [0]; MAX_EPOCHS = 2

all_results = run_all(EXPERIMENTS)
print("\nAll experiments complete!")


## 8. Results & Visualisation

In [ ]:
def load_results(results_dir):
    results = {}
    for p in Path(results_dir).rglob('result.json'):
        with open(p) as f: r = json.load(f)
        results.setdefault(r['name'], {})[r['fold_id']] = r['test_ap']
    return results

try:    _ = all_results
except NameError: all_results = load_results(RESULTS_DIR)

print(f"{'Model':<28} {'Fold A':>8} {'Fold B':>8} {'Fold C':>8} {'Mean':>8} {'Std':>7}")
print("─" * 65)
summary = {}
for name, fold_aps in all_results.items():
    aps  = [fold_aps.get(fid, float('nan')) for fid in FOLD_IDS]
    mean, std = np.nanmean(aps), np.nanstd(aps)
    summary[name] = {"aps": aps, "mean": mean, "std": std}
    print(f"{name:<28} {aps[0]:>8.4f} {aps[1]:>8.4f} {aps[2]:>8.4f} {mean:>8.4f} {std:>7.4f}")
print("─" * 65)
print(f"{'Paper UTAE 12-fold':<28} {'—':>8} {'—':>8} {'—':>8} {'0.3720':>8} {'0.0880':>7}")


In [ ]:
# Bar chart + per-fold lines (matches reference visualisation style)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
names  = list(summary.keys())
means  = [summary[n]['mean'] for n in names]
stds   = [summary[n]['std']  for n in names]
colors = plt.cm.Set2(np.linspace(0, 1, len(names)))

ax1.bar(np.arange(len(names)), means, yerr=stds, capsize=5,
        color=colors, alpha=0.85, edgecolor='black')
ax1.axhline(0.372, color='red', ls='--', lw=2, label='Paper UTAE SOTA 0.372')
ax1.set_xticks(np.arange(len(names)))
ax1.set_xticklabels([n.replace('_', '\n') for n in names], fontsize=8)
ax1.set_ylabel('Average Precision (AP)'); ax1.set_title('Model comparison: Mean AP ± std')
ax1.legend(); ax1.set_ylim(0, max(means)*1.3 if means else 1)

for i, (name, col) in enumerate(zip(names[:4], colors)):
    ax2.plot([f.split()[0] for f in FOLD_NAMES],
             [summary[name]['aps'][j] for j in range(3)],
             'o-', label=name, color=col)
ax2.set_xlabel('Fold'); ax2.set_ylabel('AP')
ax2.set_title('Per-fold AP (top 4 models)')
ax2.legend(fontsize=8); ax2.tick_params(axis='x', rotation=10)

plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'results_comparison.png'), dpi=100, bbox_inches='tight')
plt.show()


In [ ]:
# Prediction visualisation (matches reference: Input / GT / Prob / Zoomed RGB overlay)
def visualize_predictions(lit_model, fold_id, n_samples=6, device=None):
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    lit_model = lit_model.to(device).eval()
    dm = get_datamodule(fold_id, batch_size=1); dm.setup('test')

    fig, axes = plt.subplots(n_samples, 4, figsize=(16, n_samples * 3.5))
    col_titles = ['Input: active fire', 'Ground truth', 'Prediction (prob)', 'Zoomed overlay']
    for j, t in enumerate(col_titles): axes[0, j].set_title(t, fontsize=11)

    shown = 0
    with torch.no_grad():
        for x, y, doys in dm.test_dataloader():
            if shown >= n_samples: break
            if y.sum() < 5: continue
            if isinstance(lit_model, WildfireLightningModule):
                pm, plv, _ = lit_model.model(x.to(device))
            else:
                pm = torch.sigmoid(lit_model(x.to(device), doys.to(device))).unsqueeze(1)
            input_af = x[0, -1, 38].numpy()   # active fire binary channel
            gt       = y[0].numpy()
            prob     = pm[0, 0].cpu().numpy()

            axes[shown, 0].imshow(input_af, cmap='hot', vmin=0, vmax=1); axes[shown, 0].axis('off')
            axes[shown, 1].imshow(gt,   cmap='Reds',      vmin=0, vmax=1); axes[shown, 1].axis('off')
            im = axes[shown, 2].imshow(prob, cmap='hot', vmin=0, vmax=1)
            plt.colorbar(im, ax=axes[shown, 2], fraction=0.046, pad=0.04)
            axes[shown, 2].axis('off')

            # RGB overlay (green=GT, red=pred, yellow=both) — same as reference
            combined = (input_af > 0) | (gt > 0) | (prob > 0.3)
            rows_f, cols_f = np.where(combined.any(axis=1))[0], np.where(combined.any(axis=0))[0]
            if len(rows_f) and len(cols_f):
                pad = 20
                r0 = max(0, rows_f[0]-pad);   r1 = min(gt.shape[0], rows_f[-1]+pad)
                c0 = max(0, cols_f[0]-pad);   c1 = min(gt.shape[1], cols_f[-1]+pad)
                overlay = np.zeros((r1-r0, c1-c0, 3))
                overlay[:,:,1] = gt[r0:r1, c0:c1]        # green = GT
                overlay[:,:,0] = prob[r0:r1, c0:c1]      # red   = prediction
                axes[shown, 3].imshow(overlay)
                axes[shown, 1].add_patch(
                    Rectangle((c0,r0), c1-c0, r1-r0, lw=2, edgecolor='lime', facecolor='none'))
            else:
                axes[shown, 3].text(0.5, 0.5, 'No fire', ha='center', va='center',
                                    transform=axes[shown, 3].transAxes, color='gray')
            axes[shown, 3].axis('off')
            shown += 1

    plt.suptitle(f'Predictions on test set — fold {fold_id}\n(Overlay: green=GT, red=pred, yellow=both)',
                 fontsize=13, y=1.02)
    plt.tight_layout()
    plt.savefig(os.path.join(RESULTS_DIR, f'predictions_fold{fold_id}.png'), dpi=100, bbox_inches='tight')
    plt.show()
    lit_model.train()

# Example usage — load checkpoint first:
# model = WildfireTransformer(encoder_type="cnn")
# ckpt  = os.path.join(RESULTS_DIR, "E1_CNN_focal_het", "fold_0", "best.ckpt")
# lit   = WildfireLightningModule.load_from_checkpoint(ckpt, model=model)
# visualize_predictions(lit, fold_id=0)
print("visualize_predictions() defined.")


In [ ]:
# Temporal attention map: which day does the model attend to at each scale?
def visualize_attention(lit_model, fold_id, n_samples=2, device=None):
    device = device or ('cuda' if torch.cuda.is_available() else 'cpu')
    lit_model = lit_model.to(device).eval()
    dm = get_datamodule(fold_id, batch_size=1); dm.setup('test')
    shown = 0
    for x, y, doys in dm.test_dataloader():
        if shown >= n_samples: break
        if y.sum() < 5: continue
        with torch.no_grad():
            _, _, attn_maps = lit_model.model(x.to(device))
        fig, axes = plt.subplots(2, 4, figsize=(16, 8))
        for s in range(4):
            attn = attn_maps[s][0].cpu().numpy()   # (H, W, T)
            per_day = attn.mean(0).mean(0)
            axes[0, s].bar(range(N_DAYS), per_day, color='#2196F3')
            axes[0, s].set_title(f"Scale {s+1}\nMean attn/day", fontsize=9)
            axes[0, s].set_xlabel("Day (0=oldest)")
            axes[1, s].imshow(attn[:, :, per_day.argmax()], cmap='hot')
            axes[1, s].set_title(f"Attn map (day {per_day.argmax()})", fontsize=9)
            axes[1, s].axis('off')
        plt.suptitle(f'Temporal Attention — fold {fold_id}, sample {shown}', fontsize=11)
        plt.tight_layout()
        plt.savefig(os.path.join(RESULTS_DIR, f'attn_fold{fold_id}_s{shown}.png'), dpi=100, bbox_inches='tight')
        plt.show(); shown += 1
    lit_model.train()

print("visualize_attention() defined.")


## 9. Save Best Models to Drive

In [ ]:
# Save best checkpoint for each experiment to Drive (matches reference save pattern)
save_dir = os.path.join(RESULTS_DIR, 'checkpoints')
os.makedirs(save_dir, exist_ok=True)

for exp_name in all_results:
    for fold_id in FOLD_IDS:
        result_path = os.path.join(RESULTS_DIR, exp_name, f'fold_{fold_id}', 'result.json')
        if not os.path.exists(result_path):
            continue
        with open(result_path) as f:
            r = json.load(f)
        ckpt_src = r.get('ckpt', '')
        if ckpt_src and os.path.exists(ckpt_src):
            ckpt_dst = os.path.join(save_dir, f'{exp_name}_fold{fold_id}.ckpt')
            shutil.copy2(ckpt_src, ckpt_dst)
            print(f"Saved: {os.path.basename(ckpt_dst)}  (AP={r['test_ap']:.4f})")

# Save results JSON to Drive
results_json_path = os.path.join(RESULTS_DIR, 'all_results.json')
with open(results_json_path, 'w') as f:
    json.dump(all_results, f, indent=2)
print(f"\nAll results saved to: {results_json_path}")


## 9. Architecture B — VQ-VAE + Latent Diffusion

Two-stage approach inspired by STLDM:
1. **VQ-VAE** (pretrained on all data) — learns a compressed 32×32 latent space
2. **Conditioning network + Diffusion** — deterministic forecast + stochastic refinement in latent space

Run independently via `run_all_diffusion()` — separate from the main ablation loop.


### 9a. VQ-VAE

In [ ]:
from vector_quantize_pytorch import VectorQuantize

class ResBlock2d(nn.Module):
    def __init__(self, ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(ch, ch, 3, padding=1), nn.BatchNorm2d(ch), nn.GELU(),
            nn.Conv2d(ch, ch, 3, padding=1), nn.BatchNorm2d(ch),
        )
    def forward(self, x): return nn.functional.gelu(x + self.net(x))


class WildfireVQVAE(nn.Module):
    """
    Encodes 40-channel wildfire frames into a discrete 32x32 latent space.
    4x spatial downsampling: 128x128 -> 32x32.
    Consistent with Architecture A input (same 40-channel preprocessed tensor).
    """
    IN_CH = 40
    LATENT_DIM = 256
    CODEBOOK_SIZE = 512

    def __init__(self):
        super().__init__()
        ld = self.LATENT_DIM
        self.encoder = nn.Sequential(
            nn.Conv2d(self.IN_CH, 64, 4, stride=2, padding=1), nn.BatchNorm2d(64), nn.GELU(),
            ResBlock2d(64),
            nn.Conv2d(64, 128, 4, stride=2, padding=1), nn.BatchNorm2d(128), nn.GELU(),
            ResBlock2d(128),
            nn.Conv2d(128, ld, 1),
        )
        self.vq = VectorQuantize(dim=ld, codebook_size=self.CODEBOOK_SIZE,
                                 decay=0.99, commitment_weight=0.25, use_cosine_sim=True)
        self.decoder = nn.Sequential(
            nn.Conv2d(ld, 128, 1),
            ResBlock2d(128),
            nn.ConvTranspose2d(128, 64, 4, stride=2, padding=1), nn.BatchNorm2d(64), nn.GELU(),
            ResBlock2d(64),
            nn.ConvTranspose2d(64, self.IN_CH, 4, stride=2, padding=1),
        )

    def encode(self, x):
        """x: [B, 40, H, W] -> z_e: [B, ld, H/4, W/4]"""
        return self.encoder(x)

    def quantize(self, z_e):
        """z_e: [B, D, h, w] -> z_q, indices, loss"""
        B, D, h, w = z_e.shape
        z_flat = z_e.permute(0, 2, 3, 1).reshape(B*h*w, D)
        z_q_flat, indices, vq_loss = self.vq(z_flat)
        z_q = z_q_flat.reshape(B, h, w, D).permute(0, 3, 1, 2)
        return z_q, indices, vq_loss

    def decode(self, z_q):
        return self.decoder(z_q)

    def forward(self, x):
        z_e = self.encode(x)
        z_q, indices, vq_loss = self.quantize(z_e)
        x_hat = self.decode(z_q)
        return x_hat, vq_loss, z_q

print("WildfireVQVAE defined. Latent shape per frame: 32×32×256")


### 9b. Conditioning Network

In [ ]:
class ConditioningNetwork(nn.Module):
    """
    Deterministic forecast: given T latent frames, predict next-frame latent.
    Lightweight spatial-then-temporal transformer.
    """
    def __init__(self, latent_dim=256, n_heads=8, n_layers=3, max_T=10):
        super().__init__()
        self.pos_embed = nn.Embedding(max_T, latent_dim)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=latent_dim, nhead=n_heads,
            dim_feedforward=latent_dim * 4, dropout=0.1, batch_first=True,
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.out_proj = nn.Sequential(
            nn.Conv2d(latent_dim, latent_dim, 3, padding=1),
            nn.BatchNorm2d(latent_dim), nn.GELU(),
            nn.Conv2d(latent_dim, latent_dim, 1),
        )

    def forward(self, z_seq):
        """
        z_seq: [B, T, D, h, w]
        Returns z_cond: [B, D, h, w]
        """
        B, T, D, h, w = z_seq.shape
        pos = self.pos_embed(torch.arange(T, device=z_seq.device))  # [T, D]
        z_seq = z_seq + pos[None, :, :, None, None]
        # Per-spatial-location temporal attention
        x = z_seq.permute(0, 3, 4, 1, 2).reshape(B * h * w, T, D)
        x = self.transformer(x)           # [B*h*w, T, D]
        x = x[:, -1]                      # take last token
        x = x.reshape(B, h, w, D).permute(0, 3, 1, 2)  # [B, D, h, w]
        return self.out_proj(x)

print("ConditioningNetwork defined.")


### 9c. Latent Diffusion Model (DDPM)

In [ ]:
import math as _math

def _sinusoidal_embed(t, dim):
    """t: [B] -> [B, dim]"""
    half = dim // 2
    freq = torch.exp(-_math.log(10000) * torch.arange(half, device=t.device) / (half - 1))
    args = t[:, None].float() * freq[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)


class TimeEmbedMLP(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(dim, dim * 4), nn.SiLU(), nn.Linear(dim * 4, dim))
        self.dim = dim
    def forward(self, t): return self.net(_sinusoidal_embed(t, self.dim))


class ConvBlock(nn.Module):
    def __init__(self, in_ch, out_ch, t_dim):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.SiLU(),
            nn.Conv2d(out_ch, out_ch, 3, padding=1), nn.GroupNorm(8, out_ch), nn.SiLU(),
        )
        self.t_proj = nn.Linear(t_dim, out_ch)
    def forward(self, x, t_emb):
        return self.conv(x) + self.t_proj(t_emb)[:, :, None, None]


class LatentUNet(nn.Module):
    """Small U-Net denoiser operating in 32×32 latent space."""
    CHANNELS = [256, 384, 512]

    def __init__(self, in_ch, out_ch, t_dim=128):
        super().__init__()
        ch = self.CHANNELS
        self.t_mlp = TimeEmbedMLP(t_dim)
        self.down = nn.ModuleList([
            ConvBlock(in_ch,  ch[0], t_dim),
            ConvBlock(ch[0],  ch[1], t_dim),
            ConvBlock(ch[1],  ch[2], t_dim),
        ])
        self.pools = nn.ModuleList([nn.AvgPool2d(2), nn.AvgPool2d(2)])
        self.mid = ConvBlock(ch[2], ch[2], t_dim)
        self.up = nn.ModuleList([
            ConvBlock(ch[2] + ch[1], ch[1], t_dim),
            ConvBlock(ch[1] + ch[0], ch[0], t_dim),
        ])
        self.ups_conv = nn.ModuleList([
            nn.ConvTranspose2d(ch[2], ch[2], 2, stride=2),
            nn.ConvTranspose2d(ch[1], ch[1], 2, stride=2),
        ])
        self.head = nn.Conv2d(ch[0], out_ch, 1)

    def forward(self, x, t):
        t_emb = self.t_mlp(t)
        skips = []
        for i, blk in enumerate(self.down):
            x = blk(x, t_emb); skips.append(x)
            if i < len(self.pools): x = self.pools[i](x)
        x = self.mid(x, t_emb)
        for i, (blk, up) in enumerate(zip(self.up, self.ups_conv)):
            x = up(x)
            x = blk(torch.cat([x, skips[-(i+2)]], dim=1), t_emb)
        return self.head(x)


class LatentDiffusionModel(nn.Module):
    """DDPM noise predictor in latent space, conditioned on z_cond."""
    T_DIFF = 1000

    def __init__(self, latent_dim=256, t_dim=128):
        super().__init__()
        # 2×latent_dim input: noisy z_t + conditioning z_cond (concatenated)
        self.unet = LatentUNet(in_ch=latent_dim * 2, out_ch=latent_dim, t_dim=t_dim)
        betas = torch.linspace(1e-4, 0.02, self.T_DIFF)
        alphas = 1 - betas
        alphas_cum = torch.cumprod(alphas, 0)
        self.register_buffer('betas', betas)
        self.register_buffer('alphas_cum', alphas_cum)
        self.register_buffer('sqrt_ac', alphas_cum.sqrt())
        self.register_buffer('sqrt_1mac', (1 - alphas_cum).sqrt())

    def q_sample(self, z0, t, noise=None):
        """Forward diffusion: add noise at step t."""
        noise = noise or torch.randn_like(z0)
        s_ac  = self.sqrt_ac[t][:, None, None, None]
        s_1mc = self.sqrt_1mac[t][:, None, None, None]
        return s_ac * z0 + s_1mc * noise, noise

    def forward(self, z0, z_cond):
        """Training: random t, predict noise."""
        B = z0.shape[0]
        t = torch.randint(0, self.T_DIFF, (B,), device=z0.device)
        z_noisy, noise = self.q_sample(z0, t)
        eps_pred = self.unet(torch.cat([z_noisy, z_cond], dim=1), t)
        return nn.functional.mse_loss(eps_pred, noise)

    @torch.no_grad()
    def ddim_sample(self, z_cond, n_steps=50):
        """Fast DDIM sampling from noise, conditioned on z_cond."""
        B, D, h, w = z_cond.shape
        z = torch.randn(B, D, h, w, device=z_cond.device)
        step_ids = torch.linspace(self.T_DIFF - 1, 0, n_steps, dtype=torch.long, device=z_cond.device)
        for i, t_id in enumerate(step_ids):
            t_batch = t_id.expand(B)
            eps = self.unet(torch.cat([z, z_cond], dim=1), t_batch)
            ac_t  = self.alphas_cum[t_id]
            ac_t1 = self.alphas_cum[step_ids[i+1]] if i + 1 < len(step_ids) else torch.tensor(1.0, device=z.device)
            z0_hat = (z - (1 - ac_t).sqrt() * eps) / ac_t.sqrt()
            z0_hat = z0_hat.clamp(-3, 3)
            z = ac_t1.sqrt() * z0_hat + (1 - ac_t1).sqrt() * eps
        return z

print("LatentDiffusionModel defined.")


### 9d. Training Wrappers — VQ-VAE & Diffusion

In [ ]:
# ── VQ-VAE Lightning module ─────────────────────────────────────────────────
class VQVAELightning(pl.LightningModule):
    def __init__(self, lr=1e-4):
        super().__init__()
        self.vqvae = WildfireVQVAE()
        self.lr = lr
        # Weight fire channel (last channel, index 39) much higher
        w = torch.ones(40); w[-1] = 10.0; w[-2] = 5.0  # fire + active fire
        self.register_buffer('ch_weights', w)

    def _recon_loss(self, x_hat, x):
        diff = (x_hat - x) ** 2
        return (diff * self.ch_weights[None, :, None, None]).mean()

    def training_step(self, batch, _):
        x, _, __ = batch  # only need input frames; batch is (x, y, doys)
        # Use last timestep as reconstruction target (full 40ch)
        x_last = x[:, -1]  # [B, 40, H, W]
        x_hat, vq_loss, _ = self.vqvae(x_last)
        loss = self._recon_loss(x_hat, x_last) + vq_loss
        self.log('train_recon_loss', loss, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    def validation_step(self, batch, _):
        x, _, __ = batch
        x_last = x[:, -1]
        x_hat, vq_loss, _ = self.vqvae(x_last)
        loss = self._recon_loss(x_hat, x_last) + vq_loss
        self.log('val_recon_loss', loss, prog_bar=True)
        return loss

    def configure_optimizers(self):
        return torch.optim.Adam(self.parameters(), lr=self.lr)


# ── Diffusion Lightning module ────────────────────────────────────────────────
class DiffusionLightning(pl.LightningModule):
    """
    Trains ConditioningNetwork + (optionally) LatentDiffusionModel.
    mode="cond_only" → B1 (deterministic forecast only)
    mode="full"      → B2 (CondNet + DDPM)
    VQ-VAE is frozen.
    """
    def __init__(self, vqvae, mode="full", lr=1e-4):
        super().__init__()
        self.vqvae = vqvae
        for p in self.vqvae.parameters(): p.requires_grad_(False)
        self.cond_net = ConditioningNetwork()
        self.mode = mode
        self.lr = lr
        if mode == "full":
            self.diffusion = LatentDiffusionModel()
        self.val_ap = AveragePrecision(task="binary")
        self.test_ap = AveragePrecision(task="binary")

    def _encode_seq(self, x_seq):
        """x_seq: [B, T, 40, H, W] -> z_seq: [B, T, D, h, w]"""
        B, T, C, H, W = x_seq.shape
        x_flat = x_seq.reshape(B * T, C, H, W)
        with torch.no_grad():
            z_flat = self.vqvae.encode(x_flat)
            z_q, _, _ = self.vqvae.quantize(z_flat)
        return z_q.reshape(B, T, *z_q.shape[1:])

    def _predict_fire_prob(self, x_seq):
        """Full forward: encode, condition, decode, extract fire channel."""
        z_seq   = self._encode_seq(x_seq)
        z_input = z_seq[:, :-1]          # first T-1 frames as context
        z_target = z_seq[:, -1]          # last frame as target
        z_cond  = self.cond_net(z_input)

        if self.mode == "full":
            # Sample from diffusion
            z_pred = self.diffusion.ddim_sample(z_cond, n_steps=50)
        else:
            z_pred = z_cond

        x_pred = self.vqvae.decode(z_pred)  # [B, 40, H, W]
        fire_pred = torch.sigmoid(x_pred[:, -1])  # fire channel
        return fire_pred, z_target, z_cond

    def training_step(self, batch, _):
        x, y, _ = batch
        z_seq   = self._encode_seq(x)
        z_input = z_seq[:, :-1]
        z_target = z_seq[:, -1]
        z_cond  = self.cond_net(z_input)

        if self.mode == "full":
            loss = self.diffusion(z_target, z_cond)
        else:
            # Deterministic: MSE in latent space
            loss = nn.functional.mse_loss(z_cond, z_target)
        self.log('train_loss', loss, prog_bar=True, on_epoch=True, on_step=False)
        return loss

    def validation_step(self, batch, _):
        x, y, _ = batch
        fire_pred, z_target, _ = self._predict_fire_prob(x)
        loss = nn.functional.binary_cross_entropy(fire_pred, y.float())
        self.val_ap(fire_pred, y)
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_ap',   self.val_ap, prog_bar=True)
        return loss

    def test_step(self, batch, _):
        x, y, _ = batch
        fire_pred, _, _ = self._predict_fire_prob(x)
        loss = nn.functional.binary_cross_entropy(fire_pred, y.float())
        self.test_ap(fire_pred, y)
        self.log('test_loss', loss)
        self.log('test_ap',   self.test_ap)
        return loss

    def on_test_epoch_end(self):
        ap = self.test_ap.compute()
        print(f"  Test AP (diffusion/{self.mode}): {ap.item():.4f}")

    def configure_optimizers(self):
        params = list(self.cond_net.parameters())
        if self.mode == "full":
            params += list(self.diffusion.parameters())
        return torch.optim.AdamW(params, lr=self.lr, weight_decay=0.01)

print("VQVAELightning + DiffusionLightning defined.")


### 9e. Pretrain VQ-VAE

In [ ]:
def pretrain_vqvae(fold_id=0, max_epochs=30, lr=1e-4):
    """
    Pretrain VQ-VAE on ALL years (use fold_id=0 data — all 4 years present across splits).
    We train on train+val+test combined since VQ-VAE is self-supervised (reconstruction only).
    For simplicity, just use fold_id=0 train split (2018+2019) — enough to learn codebook.
    Returns trained vqvae (nn.Module).
    """
    ckpt_path = os.path.join(RESULTS_DIR, "vqvae", "vqvae_best.ckpt")
    enc_path  = os.path.join(RESULTS_DIR, "vqvae", "vqvae_encoder.pt")

    if os.path.exists(enc_path):
        print(f"VQ-VAE already trained. Loading from {enc_path}")
        lit = VQVAELightning()
        state = torch.load(enc_path, map_location='cpu')
        lit.vqvae.load_state_dict(state)
        return lit.vqvae

    os.makedirs(os.path.join(RESULTS_DIR, "vqvae"), exist_ok=True)
    dm = get_datamodule(fold_id=0, batch_size=BATCH_SIZE)
    lit = VQVAELightning(lr=lr)

    ckpt_cb = ModelCheckpoint(
        dirpath=os.path.join(RESULTS_DIR, "vqvae"),
        filename="vqvae_best", monitor="val_recon_loss", mode="min", save_top_k=1,
    )
    trainer = pl.Trainer(
        max_epochs=max_epochs, accelerator="auto",
        precision="16-mixed" if torch.cuda.is_available() else "32-true",
        callbacks=[ckpt_cb], log_every_n_steps=10,
        enable_progress_bar=True, enable_model_summary=False, logger=False,
    )
    print("\nPretraining VQ-VAE...")
    trainer.fit(lit, datamodule=dm)

    # Save just the vqvae state dict for reuse
    torch.save(lit.vqvae.state_dict(), enc_path)
    print(f"VQ-VAE saved to {enc_path}")
    return lit.vqvae

print("pretrain_vqvae() defined.")


### 9f. Diffusion Experiment Runner

In [ ]:
def run_experiment_diffusion(name, mode, vqvae, fold_id, fold_name, lr=1e-4):
    """Train DiffusionLightning on one fold. Returns test AP."""
    ckpt_dir = os.path.join(RESULTS_DIR, name, f"fold_{fold_id}")
    os.makedirs(ckpt_dir, exist_ok=True)
    dm = get_datamodule(fold_id)

    lit = DiffusionLightning(vqvae=vqvae, mode=mode, lr=lr)

    ckpt_cb = ModelCheckpoint(
        dirpath=ckpt_dir, filename="best",
        monitor="val_loss", mode="min", save_top_k=1,
    )
    trainer = pl.Trainer(
        max_epochs=MAX_EPOCHS, accelerator="auto",
        precision="16-mixed" if torch.cuda.is_available() else "32-true",
        callbacks=[ckpt_cb], log_every_n_steps=10,
        enable_progress_bar=True, enable_model_summary=False, logger=False,
    )

    print(f"\n{'='*60}")
    print(f"Experiment : {name}  ({mode})")
    print(f"Fold       : {fold_id} — {fold_name}")
    print(f"{'='*60}")

    trainer.fit(lit, datamodule=dm)
    results = trainer.test(lit, datamodule=dm, ckpt_path="best")
    ap = float(results[0].get("test_ap", 0.0))

    out = {"name": name, "fold_id": fold_id, "fold_name": fold_name,
           "test_ap": ap, "ckpt": ckpt_cb.best_model_path}
    with open(os.path.join(ckpt_dir, "result.json"), "w") as f:
        json.dump(out, f, indent=2)
    return ap


def run_all_diffusion(vqvae=None):
    """
    Separate runner for Architecture B experiments.
    Kept separate from run_all() so diffusion failures don\'t affect
    the main ablation results.

    B1 = VQ-VAE + CondNet only (deterministic latent forecast)
    B2 = VQ-VAE + CondNet + DDPM (stochastic diffusion)
    E2 = Architecture A with CNN encoder pretrained from VQ-VAE
    """
    # Step 1: pretrain VQ-VAE (shared across all diffusion experiments)
    if vqvae is None:
        vqvae = pretrain_vqvae(fold_id=0, max_epochs=30)

    DIFF_EXPERIMENTS = [
        {"name": "B1_cond_only",  "mode": "cond_only"},  # deterministic latent forecast
        {"name": "B2_full_diff",  "mode": "full"},        # + DDPM stochastic refinement
    ]

    all_diff_results = {}

    # ── B1 / B2 ──────────────────────────────────────────────────────────────
    for exp in DIFF_EXPERIMENTS:
        all_diff_results[exp["name"]] = {}
        for fold_id, fold_name in zip(FOLD_IDS, FOLD_NAMES):
            try:
                ap = run_experiment_diffusion(
                    exp["name"], exp["mode"], vqvae, fold_id, fold_name,
                )
                all_diff_results[exp["name"]][fold_id] = ap
            except Exception as e:
                print(f"  [ERROR] {exp[\'name\']} fold {fold_id}: {e}")
                all_diff_results[exp["name"]][fold_id] = None

    # ── E2: Architecture A with pretrained CNN encoder ────────────────────────
    # Load VQ-VAE encoder weights into CNNEncoder (all layers except first conv)
    all_diff_results["E2_CNN_pretrained"] = {}
    for fold_id, fold_name in zip(FOLD_IDS, FOLD_NAMES):
        try:
            model = WildfireTransformer(encoder_type="cnn")
            # Transfer VQ-VAE encoder weights (skip first conv: 40ch VQ-VAE vs 18ch CNN)
            vqvae_sd = vqvae.encoder.state_dict()
            cnn_sd   = model.encoder.state_dict()
            matched = {k: v for k, v in vqvae_sd.items()
                       if k in cnn_sd and v.shape == cnn_sd[k].shape}
            cnn_sd.update(matched)
            model.encoder.load_state_dict(cnn_sd, strict=False)
            print(f"  E2 fold {fold_id}: transferred {len(matched)}/{len(cnn_sd)} encoder layers")
            ap = run_experiment("E2_CNN_pretrained", model, fold_id, fold_name,
                                loss_type="focal_het")
            all_diff_results["E2_CNN_pretrained"][fold_id] = ap
        except Exception as e:
            print(f"  [ERROR] E2 fold {fold_id}: {e}")
            all_diff_results["E2_CNN_pretrained"][fold_id] = None

    diff_path = os.path.join(RESULTS_DIR, "all_diffusion_results.json")
    with open(diff_path, "w") as f:
        json.dump(all_diff_results, f, indent=2)
    print(f"\nDiffusion results saved to: {diff_path}")
    return all_diff_results

print("run_all_diffusion() defined.")


## 10. Run Diffusion Experiments

In [ ]:
# Run Architecture B (VQ-VAE + Diffusion) and E2 (pretrained CNN).
# Separated from run_all() so a crash here doesn\'t lose main ablation results.
#
# Runtime estimate on Colab A100:
#   VQ-VAE pretrain:    ~15 min  (30 epochs)
#   B1 (cond only):     ~20 min  × 3 folds
#   B2 (full diffusion): ~40 min × 3 folds
#   E2 (pretrained CNN): ~40 min × 3 folds
#   Total: ~4-5 h

# Uncomment to run:
# diff_results = run_all_diffusion()

# Or run individual pieces:
# vqvae = pretrain_vqvae(fold_id=0, max_epochs=30)
# diff_results = run_all_diffusion(vqvae=vqvae)  # skips re-pretraining
print("Architecture B cells loaded. Uncomment run_all_diffusion() to execute.")


## 11. Combined Results Table

In [ ]:
import os, json
from pathlib import Path

def load_all_results(results_dir=RESULTS_DIR):
    """Load all result.json files into a flat dict: {exp_name: {fold_id: ap}}"""
    results = {}
    for p in Path(results_dir).rglob("result.json"):
        with open(p) as f: r = json.load(f)
        exp   = r.get("name",    p.parts[-3])
        fold  = r.get("fold_id", int(p.parts[-2].replace("fold_","")))
        ap    = r.get("test_ap", None)
        results.setdefault(exp, {})[fold] = ap
    return results

def print_results_table(results):
    """Print mean±std AP per experiment across folds."""
    import numpy as np
    rows = []
    for name, fold_dict in sorted(results.items()):
        vals = [v for v in fold_dict.values() if v is not None]
        if not vals: continue
        mean, std = np.mean(vals), np.std(vals)
        rows.append((name, mean, std, vals))
    rows.sort(key=lambda r: -r[1])

    print(f"{'Experiment':<28}  {'AP mean':>8}  {'AP std':>7}  Folds")
    print("-" * 60)
    for name, mean, std, vals in rows:
        fold_str = "  ".join(f"{v:.4f}" for v in vals)
        print(f"{name:<28}  {mean:.4f}    {std:.4f}   {fold_str}")

    # Paper UTAE reference
    print("-" * 60)
    print(f"{'[Paper UTAE veg-best]':<28}  0.3720    0.0880   (from Table 5)")

try:
    combined = load_all_results()
    print_results_table(combined)
except Exception as e:
    print(f"No results yet: {e}")
